In [86]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re

In [46]:
data = pd.read_csv('data/online_retail_II.csv')
data.shape

(1067371, 8)

### Business question: - Refine Later ###
* “How does price impact demand, and what price maximizes revenue (or profit)?”
*  What is the optimal price point?
*  How sensitive are customers to price changes?
*  Do promotions actually increase revenue or just volume?

### Project Outline - Refine Later ###
1. EDA
    * Price distributions
    * Demand distibution (p vs. Q)
    * Promo comperitors
2. DGP
    * Endogeniety? 
    * Simultaneity? 
    * Promo treatment effects? 
3. Cleaning/Preprocessing
4. Feature Eng
    * Demand Signals
        * Rolling Ave
    * Price in context
    * Market in context
5. Elasticity Estimation
    * log-OLS
6. Addressing Endogeneity in Pricing
    * FE estimation
    * IV estimation
7. Validation
8. RevOpt
    * Simulate Rev/Profit curves
    * Identify Max and regionality within P/Q
9. A/B testing promos
    * OlS
    * DiD
10. Elasticity by segment
11. Demand & Revenue Curve Visualization
12. Streamlit? 


### Data Description ###
* InvoiceNo: Invoice number. Nominal. A 6-digit integral number uniquely assigned to each transaction. If this code starts with the letter 'c', it indicates a cancellation.
* StockCode: Product (item) code. Nominal. A 5-digit integral number uniquely assigned to each distinct product.
* Description: Product (item) name. Nominal.
* Quantity: The quantities of each product (item) per transaction. Numeric.
* InvoiceDate: Invoice date and time. Numeric. The day and time when a transaction was generated.
* Price: Unit price. Numeric. Product price per unit in sterling (Â£).
* CustomerID: Customer number. Nominal. A 5-digit integral number uniquely assigned to each customer.
* Country: Country name. Nominal. The name of the country where a customer resides.

In [49]:
data['Refund'] = np.where(data['Price'] < 0, 1, 0)
data['Return'] = np.where(data['Quantity'] < 0, 1, 0 )
returned_sales = data[(data['Return'] == 1) | (data['Refund'] == 1)]


In [43]:
returned_sales['Description'].value_counts(ascending=False).head(10)

Description
Manual                                537
REGENCY CAKESTAND 3 TIER              347
POSTAGE                               229
BAKING SET 9 PIECE RETROSPOT          211
STRAWBERRY CERAMIC TRINKET BOX        184
Discount                              172
WHITE HANGING HEART T-LIGHT HOLDER    135
check                                 123
WHITE CHERRY LIGHTS                   121
RED RETROSPOT CAKE STAND              111
Name: count, dtype: int64

In [50]:
sales_data = data[(data['Refund'] == 0) & (data['Return'] == 0)]
sales_data.shape

(1044416, 10)

In [79]:
sales_data['Clean_Desc'] = (
    sales_data['Description']
    .fillna("")
    .str.upper()
    .str.replace(r"[^A-Z\s]", " ", regex = True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [83]:
top_desc = sales_data['Clean_Desc'].value_counts(ascending= False).head(25)
print("Top full descriptions:")
print(top_desc)


Top full descriptions:
Clean_Desc
WHITE HANGING HEART T LIGHT HOLDER    5783
REGENCY CAKESTAND TIER                4065
JUMBO BAG RED RETROSPOT               3395
ASSORTED COLOUR BIRD ORNAMENT         2939
PARTY BUNTING                         2742
LUNCH BAG BLACK SKULL                 2484
LUNCH BAG SUKI DESIGN                 2473
STRAWBERRY CERAMIC TRINKET BOX        2429
JUMBO STORAGE BAG SUKI                2400
FRENCH BLUE METAL DOOR SIGN           2303
HEART OF WICKER SMALL                 2299
JUMBO SHOPPER VINTAGE RED PAISLEY     2275
TEATIME FAIRY CAKE CASES              2257
PAPER CHAIN KIT S CHRISTMAS           2198
REX CASH CARRY JUMBO SHOPPER          2190
LUNCH BAG SPACEBOY DESIGN             2186
HOME BUILDING BLOCK WORD              2172
WOODEN FRAME ANTIQUE WHITE            2151
LUNCH BAG CARS BLUE                   2141
NATURAL SLATE HEART CHALKBOARD        2119
HEART OF WICKER LARGE                 2090
WOODEN PICTURE FRAME WHITE FINISH     2085
PACK OF PINK PAISLEY

In [84]:

all_words = (
    sales_data['Clean_Desc']
    .str.split()
    .explode()
)

stopwords = {
    "THE", "AND", "OF", "IN", "TO", "FOR", "ON", "WITH", "A", "AN",
    "SET", "PACK", "SMALL", "LARGE", "ASSORTED"
}

word_counts = (
    all_words[
        all_words.notna()
        & (all_words.str.len() > 2)
        & (~all_words.isin(stopwords))
    ]
    .value_counts()
    .head(50)
)

print("\nTop single words:")
print(word_counts)



Top single words:
Clean_Desc
BAG           91439
RED           91005
HEART         78132
PINK          64048
RETROSPOT     57376
VINTAGE       54861
DESIGN        53286
WHITE         49966
BOX           49755
METAL         45041
CAKE          44892
CHRISTMAS     44073
BLUE          41588
HANGING       36187
LIGHT         35746
SIGN          35226
JUMBO         34759
HOLDER        34241
PAPER         30608
LUNCH         29986
GLASS         26645
TEA           26316
CARD          25569
DECORATION    24780
WOODEN        23468
CASES         23185
BOTTLE        22943
SPOTTY        22226
HOT           21839
WATER         21171
ROSE          20527
CERAMIC       20244
SPACEBOY      18267
MUG           18060
PAISLEY       17931
FAIRY         17247
BLACK         16985
TIN           16923
POLKADOT      16711
CREAM         16614
HOME          16495
GREEN         16177
SPOT          16144
PARTY         15366
GARDEN        15183
FELTCRAFT     15076
MINI          15036
LOVE          14496
BUNTING   

In [85]:
def get_bigrams(text: str) -> list[str]:
    words = [w for w in text.split() if len(w) > 2 and w not in stopwords]
    return [" ".join(pair) for pair in zip(words, words[1:])]

bigram_counter = Counter()

for desc in df["Description_clean"]:
    bigram_counter.update(get_bigrams(desc))

top_bigrams = pd.Series(dict(bigram_counter)).sort_values(ascending=False).head(50)

print("\nTop bigrams:")
print(top_bigrams)

NameError: name 'Counter' is not defined